# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
# QUAN TRỌNG: các thay đổi pretrain (gen_all, vision unfreeze, whole-word mask) nằm ở
# NHÁNH exp/pretrain-gen-all — KHÔNG phải main. Không checkout đúng nhánh sẽ bị lỗi
# 'unrecognized arguments: --vision_unfreeze_last_n --mlm_mask_mode'.
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'                       # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — TỰ ĐỘNG in debug đầy đủ (không cần set env)
> ⚠️ Vision unfreeze TẮT (`--vision_unfreeze_last_n 0`), có guard NaN pretrain-only. ✅ MLM chỉ dùng câu hỏi. 📖 Gen = `read` (denoise tắt).

Mock **luôn** in debug. Trong log tìm:
1. `>>> [pretrain] ... gen_task = read`
2. `🔬 [VERIFY]` toàn ✅, `✅ [GEN] gen_loss finite & > 0`, KHÔNG có `🚨 ... has NaN`
3. Per-step `[Pretrain] ... Loss(M) Loss(I) Loss(TWC) Loss(GEN)` **giảm dần**
4. `🔎 [MLM DEBUG]`: mỗi từ mask hiện **token thô gold/pred + gộp thành word** (`gold: tokens=[...] → 'từ'` / `pred: ...`)
5. `🔧 [GEN DEBUG]`: target(clean) vs output

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
# MOCK: tự động in debug đầy đủ (không cần set env). Full run mặc định KHÔNG debug.
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm: vision unfreeze gây grad-nổ + NaN forward
    '--mlm_mask_mode', 'wholeword',    # mask trọn từ (bỏ 'nạng' copy subword)
    '--gen_task', 'read',              # read-scene-text (PreSTU-like). denoise TẮT (target=OCR nhiễu, khó bảo vệ)
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN — mặc định KHÔNG debug (chỉ progress bar + eval)
Full run **mặc định tắt debug** (chỉ thanh tiến trình train + kết quả eval trên val → tránh đầy output/lag). Muốn **bật debug** cho full: đặt `os.environ['TWC_TRAIN_LOG']='1'` trước khi gọi. Theo dõi `loss_mlm` & `loss_gen` **giảm dần** qua các lần eval.

In [ ]:
import os, importlib
os.environ.pop('TWC_TRAIN_LOG', None)   # TẮT debug per-step: full run chỉ hiện progress bar + kết quả eval
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm (vision instability)
    '--mlm_mask_mode', 'wholeword',
    '--gen_task', 'read',
    '--num_train_epochs', '3',
])

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)